# Tema 7 — NER (Listado 2) — Notebook resuelto

Este notebook resuelve **Listado II**:

- **Ejercicio 1:** Fine-tuning de NER con **spaCy** + **evaluación** (precision/recall/F1), generalización y robustez a **typos**.  
- **Ejercicio 2:** Fine-tuning de un **Transformer** (`dccuchile/bert-base-spanish-wwm-cased`) para NER biomédico en spaCy (**sequence labeling** con etiquetas IOB convertidas a spans) + evaluación.

> [!important] Recomendación
> Ejecuta en un entorno con internet (para descargar modelos) y, si puedes, GPU para el ejercicio 2.

---

## 0) Instalación (ejecutar una vez)

Descomenta lo que necesites:

```bash
pip install -U spacy spacy-transformers nltk pandas scikit-learn
python -m spacy download es_core_news_sm
```

Para el ejercicio 2 (Transformers en spaCy):
```bash
pip install -U spacy-transformers
```

---

In [3]:
# Reproducibilidad básica
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# Ejercicio 1 — Fine-tuning y evaluación de NER con spaCy

Se entrenará (fine-tuning) el componente **NER** de un modelo en español y se evaluará con:
- **Precision**, **Recall**, **F1** global
- Métricas **por tipo de entidad**

Además:
- Probar un **test más diferente** (generalización).
- Probar **robustez** con errores ortográficos en entidades (typos).

---

In [4]:
import spacy
from spacy.training import Example

# Modelo base en español
nlp = spacy.load("es_core_news_sm")

print("Pipeline:", nlp.pipe_names)
ner = nlp.get_pipe("ner")
print("Labels actuales:", ner.labels)


OSError: [E050] Can't find model 'es_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
# Helper del enunciado: (texto, [(ent_text, label), ...]) -> (texto, {"entities":[(start,end,label),...]})
def make_example(text, entities):
    """
    entities: lista de tuplas (texto_entidad, etiqueta)
    Devuelve el formato que espera spaCy:
    (texto, {"entities": [(inicio, fin, etiqueta), ...]})
    """
    spans = []
    for entity_text, label in entities:
        start = text.index(entity_text)  # asume ocurrencia única
        end = start + len(entity_text)
        spans.append((start, end, label))
    return text, {"entities": spans}


## 1.1) Construir TRAIN (texto + entidades)

Etiquetas usadas:
- `PER`: persona
- `ORG`: organización
- `LOC`: localización

---

In [ ]:
TRAIN_SIMPLE = [
    ("Pedro Sánchez se reunió con Microsoft en Madrid el lunes.",
     [("Pedro Sánchez", "PER"), ("Microsoft", "ORG"), ("Madrid", "LOC")]),
    ("La Universidad Rey Juan Carlos presentó un proyecto en Móstoles.",
     [("Universidad Rey Juan Carlos", "ORG"), ("Móstoles", "LOC")]),
    ("Ana García trabaja en Google desde 2021.",
     [("Ana García", "PER"), ("Google", "ORG")]),
    ("Apple anunció el nuevo iPhone en Barcelona.",
     [("Apple", "ORG"), ("Barcelona", "LOC")]),
    ("El Real Madrid fichó a Jude Bellingham en 2023.",
     [("Real Madrid", "ORG"), ("Jude Bellingham", "PER")]),
    ("María López visitó Sevilla durante la Semana Santa.",
     [("María López", "PER"), ("Sevilla", "LOC")]),
    ("OpenAI abrió una oficina en París.",
     [("OpenAI", "ORG"), ("París", "LOC")]),
    ("Carlos Ruiz compró un Samsung Galaxy en Valencia.",
     [("Carlos Ruiz", "PER"), ("Samsung Galaxy", "ORG"), ("Valencia", "LOC")]),
]

TRAIN_DATA = [make_example(t, ents) for t, ents in TRAIN_SIMPLE]

train_examples = []
for text, ann in TRAIN_DATA:
    doc = nlp.make_doc(text)
    train_examples.append(Example.from_dict(doc, ann))

print("Nº ejemplos de entrenamiento:", len(train_examples))
print("Ejemplo 1 ents:", [(e.text, e.label_) for e in train_examples[0].reference.ents])


## 1.2) Fine-tuning del NER

Entrenamos solo el componente `ner`.

---

In [ ]:
import random

disabled_pipes = [p for p in nlp.pipe_names if p != "ner"]
optimizer = nlp.create_optimizer()

EPOCHS = 30

with nlp.disable_pipes(*disabled_pipes):
    for epoch in range(EPOCHS):
        random.shuffle(train_examples)
        losses = {}
        for ex in train_examples:
            nlp.update([ex], sgd=optimizer, drop=0.2, losses=losses)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Losses: {losses}")


## 1.3) Predicción sobre textos nuevos (del listado)

---

In [ ]:
TEST_SENTENCES = [
    "Pedro Sánchez viajó a París para reunirse con OpenAI.",
    "Google presentó un nuevo Pixel en Madrid en 2024.",
    "Ana García visitó la Universidad Rey Juan Carlos.",
    "Microsoft organizará un evento en Barcelona el martes.",
]

for s in TEST_SENTENCES:
    doc = nlp(s)
    print("\nTEXT:", s)
    for ent in doc.ents:
        print(" -", ent.text, ent.label_)


## 1.4) Evaluación automática (precision/recall/F1 + por tipo)

Creamos anotación gold para test y evaluamos con `nlp.evaluate`.

---

In [ ]:
TEST_SIMPLE = [
    ("Pedro Sánchez viajó a París para reunirse con OpenAI.",
     [("Pedro Sánchez", "PER"), ("París", "LOC"), ("OpenAI", "ORG")]),
    ("Google presentó un nuevo Pixel en Madrid en 2024.",
     [("Google", "ORG"), ("Madrid", "LOC")]),
    ("Ana García visitó la Universidad Rey Juan Carlos.",
     [("Ana García", "PER"), ("Universidad Rey Juan Carlos", "ORG")]),
    ("Microsoft organizará un evento en Barcelona el martes.",
     [("Microsoft", "ORG"), ("Barcelona", "LOC")]),
]

TEST_DATA = [make_example(t, ents) for t, ents in TEST_SIMPLE]

test_examples = []
for text, ann in TEST_DATA:
    pred_doc = nlp(text)
    test_examples.append(Example.from_dict(pred_doc, ann))

scores = nlp.evaluate(test_examples)
print("Precision:", scores["ents_p"])
print("Recall   :", scores["ents_r"])
print("F1       :", scores["ents_f"])
print("\nPor tipo:")
print(scores["ents_per_type"])


## 1.5) Test “más diferente” (generalización)

---

In [ ]:
TEST_DIFFERENT_SIMPLE = [
    ("Ayer, en Valencia, Carlos Ruiz presentó a OpenAI un prototipo.",
     [("Valencia", "LOC"), ("Carlos Ruiz", "PER"), ("OpenAI", "ORG")]),
    ("Barcelona y Madrid compiten por atraer inversiones de Google.",
     [("Barcelona", "LOC"), ("Madrid", "LOC"), ("Google", "ORG")]),
    ("La URJC colaborará con Microsoft y Apple en un nuevo proyecto.",
     [("URJC", "ORG"), ("Microsoft", "ORG"), ("Apple", "ORG")]),
]

TEST_DIFFERENT_DATA = [make_example(t, ents) for t, ents in TEST_DIFFERENT_SIMPLE]

diff_examples = []
for text, ann in TEST_DIFFERENT_DATA:
    pred_doc = nlp(text)
    diff_examples.append(Example.from_dict(pred_doc, ann))

scores_diff = nlp.evaluate(diff_examples)
print("Precision:", scores_diff["ents_p"])
print("Recall   :", scores_diff["ents_r"])
print("F1       :", scores_diff["ents_f"])
print("\nPor tipo:")
print(scores_diff["ents_per_type"])


## 1.6) Robustez: errores ortográficos (typos)

---

In [ ]:
def typo_simple(s: str, seed=42) -> str:
    """Typo simple: swap de dos caracteres contiguos (si se puede)."""
    rng = random.Random(seed)
    if len(s) < 4:
        return s
    i = rng.randint(1, len(s) - 2)
    chars = list(s)
    chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def apply_typos_to_sample(text: str, entities):
    new_text = text
    new_entities = []
    for ent_text, label in entities:
        ent_typo = typo_simple(ent_text, seed=hash(ent_text) % (2**32))
        new_text = new_text.replace(ent_text, ent_typo, 1)
        new_entities.append((ent_typo, label))
    return new_text, new_entities

TEST_TYPO_SIMPLE = [apply_typos_to_sample(t, ents) for t, ents in TEST_SIMPLE]
TEST_TYPO_DATA = [make_example(t, ents) for t, ents in TEST_TYPO_SIMPLE]

typo_examples = []
for text, ann in TEST_TYPO_DATA:
    pred_doc = nlp(text)
    typo_examples.append(Example.from_dict(pred_doc, ann))

scores_typo = nlp.evaluate(typo_examples)

print("F1 original:", scores["ents_f"])
print("F1 typos   :", scores_typo["ents_f"])
print("\nPor tipo (typos):")
print(scores_typo["ents_per_type"])

print("\nEjemplo:")
print("ORIG:", TEST_SIMPLE[0][0])
print("TYPO:", TEST_TYPO_SIMPLE[0][0])


# Ejercicio 2 — Fine-tuning sobre Transformers (NER biomédico)

Modelo: `dccuchile/bert-base-spanish-wwm-cased`

Etiquetas:
`ENFERMEDAD`, `FARMACO`, `SINTOMA`, `PROCEDIMIENTO`, `HOSPITAL`, `FECHA`

Los datos vienen como `tokens` + `ner_tags` (IOB). Convertimos IOB → spans para spaCy.

---

In [ ]:
# Pipeline transformer + ner (spaCy)
# Requiere: pip install spacy-transformers
import spacy

try:
    nlp_trf = spacy.blank("es")
    nlp_trf.add_pipe("transformer", config={"model": {"name": "dccuchile/bert-base-spanish-wwm-cased"}})
    nlp_trf.add_pipe("ner", last=True)
    print("Pipeline TRF:", nlp_trf.pipe_names)
except Exception as e:
    print("No se pudo crear pipeline TRF.")
    print("Error:", repr(e))


In [ ]:
# Datos del enunciado
train_examples_raw = [
    {
        "tokens": ["El", "paciente", "fue", "tratado", "con", "ibuprofeno", "por", "neumonía", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "B-FARMACO", "O", "B-ENFERMEDAD", "O"]
    },
    {
        "tokens": ["María", "presenta", "fiebre", "y", "dolor", "torácico", "desde", "ayer", "."],
        "ner_tags": ["O", "O", "B-SINTOMA", "O", "B-SINTOMA", "I-SINTOMA", "O", "B-FECHA", "O"]
    },
    {
        "tokens": ["Se", "realizó", "una", "radiografía", "de", "tórax", "en", "el", "Hospital", "La", "Paz", "."],
        "ner_tags": ["O", "O", "O", "B-PROCEDIMIENTO", "I-PROCEDIMIENTO", "I-PROCEDIMIENTO", "O", "O",
                     "B-HOSPITAL", "I-HOSPITAL", "I-HOSPITAL", "O"]
    },
    {
        "tokens": ["El", "tratamiento", "con", "amoxicilina", "se", "inició", "el", "lunes", "."],
        "ner_tags": ["O", "O", "O", "B-FARMACO", "O", "O", "O", "B-FECHA", "O"]
    },
    {
        "tokens": ["El", "paciente", "fue", "diagnosticado", "de", "diabetes", "tipo", "2", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "B-ENFERMEDAD", "I-ENFERMEDAD", "I-ENFERMEDAD", "O"]
    },
]


In [ ]:
# IOB -> spans
from spacy.tokens import Doc
from spacy.training import Example

def iob_to_spans(doc: Doc, tags):
    spans = []
    start = None
    label = None

    for i, tag in enumerate(tags):
        if tag == "O":
            if start is not None:
                span = doc[start:i]
                spans.append((span.start_char, span.end_char, label))
                start, label = None, None
            continue

        prefix, ent_label = tag.split("-", 1)

        if prefix == "B":
            if start is not None:
                span = doc[start:i]
                spans.append((span.start_char, span.end_char, label))
            start = i
            label = ent_label

        elif prefix == "I":
            if start is None:
                start = i
                label = ent_label
            elif label != ent_label:
                span = doc[start:i]
                spans.append((span.start_char, span.end_char, label))
                start = i
                label = ent_label

    if start is not None:
        span = doc[start:len(tags)]
        spans.append((span.start_char, span.end_char, label))

    return spans

def raw_to_example(nlp_obj, tokens, tags):
    doc = Doc(nlp_obj.vocab, words=tokens)
    spans = iob_to_spans(doc, tags)
    return Example.from_dict(doc, {"entities": spans})


In [ ]:
# Crear examples, añadir labels, inicializar, entrenar y evaluar
import random

try:
    ner_trf = nlp_trf.get_pipe("ner")

    labels = set()
    trf_examples = []
    for ex in train_examples_raw:
        for t in ex["ner_tags"]:
            if t != "O":
                labels.add(t.split("-", 1)[1])
        trf_examples.append(raw_to_example(nlp_trf, ex["tokens"], ex["ner_tags"]))

    for lab in sorted(labels):
        ner_trf.add_label(lab)

    print("Labels:", sorted(labels))

    nlp_trf.initialize(get_examples=lambda: trf_examples)

    optimizer = nlp_trf.create_optimizer()
    EPOCHS_TRF = 10

    for epoch in range(EPOCHS_TRF):
        random.shuffle(trf_examples)
        losses = {}
        for ex in trf_examples:
            nlp_trf.update([ex], sgd=optimizer, drop=0.1, losses=losses)
        print(f"Epoch {epoch+1:02d} | Losses: {losses}")

    # Test pequeño (ampliable)
    TEST_BIO_RAW = [
        {
            "tokens": ["El", "paciente", "recibió", "paracetamol", "por", "gripe", "en", "Hospital", "La", "Paz", "ayer", "."],
            "ner_tags": ["O", "O", "O", "B-FARMACO", "O", "B-ENFERMEDAD", "O", "B-HOSPITAL", "I-HOSPITAL", "I-HOSPITAL", "B-FECHA", "O"]
        }
    ]

    gold_examples = [raw_to_example(nlp_trf, ex["tokens"], ex["ner_tags"]) for ex in TEST_BIO_RAW]
    eval_examples = []
    for gold_ex in gold_examples:
        text = gold_ex.text
        pred_doc = nlp_trf(text)
        gold = {"entities": [(e.start_char, e.end_char, e.label_) for e in gold_ex.reference.ents]}
        eval_examples.append(Example.from_dict(pred_doc, gold))

    scores_bio = nlp_trf.evaluate(eval_examples)
    print("\nPrecision:", scores_bio["ents_p"])
    print("Recall   :", scores_bio["ents_r"])
    print("F1       :", scores_bio["ents_f"])
    print("\nPor tipo:")
    print(scores_bio["ents_per_type"])

    # Predicción en texto nuevo
    new_text = "Se realizó una radiografía de tórax en el Hospital La Paz el lunes y se trató con ibuprofeno por neumonía."
    doc = nlp_trf(new_text)
    print("\nTEXT:", new_text)
    for ent in doc.ents:
        print(" -", ent.text, ent.label_)

except Exception as e:
    print("No se pudo entrenar/evaluar el modelo TRF.")
    print("Error:", repr(e))


## Qué debes comentar al entregar

**Ejercicio 1**
- Resultados en test similar vs test diferente.
- Caída de métricas con typos (robustez).

**Ejercicio 2**
- `ents_p`, `ents_r`, `ents_f` y `ents_per_type`.
- Ejemplo de predicción en texto biomédico nuevo.

---